In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/083648.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/123482.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/130301.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/157069.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/140120.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/128808.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/040257.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/076358.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/054361.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/042233.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/058266.jpg
/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/010011.jpg
/kaggle/input/datasets/therealcyberlord/

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import numpy as np

# 1. Veri Seti Hazırlığı
class CelebADataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

# Görselleri 64x64 yapıp normalleştiriyoruz
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

dataset = CelebADataset(root_dir='/kaggle/input/datasets/therealcyberlord/50k-celeba-dataset-64x64/50k/', transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# 2. VAE Model Mimarisi
class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()
        
        # Encoder: 64x64 görseli sıkıştırır
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        
        # Latent space parametreleri (Mean ve Log-Variance)
        self.fc_mu = nn.Linear(256*4*4, latent_dim)
        self.fc_logvar = nn.Linear(256*4*4, latent_dim)
        
        # Decoder: Latent temsilinden görsel üretir
        self.decoder_input = nn.Linear(latent_dim, 256*4*4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1), nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        encoded = self.encoder(x)
        mu = self.fc_mu(encoded)
        logvar = self.fc_logvar(encoded)
        z = self.reparameterize(mu, logvar)
        de_input = self.decoder_input(z).view(-1, 256, 4, 4)
        return self.decoder(de_input), mu, logvar


device = torch.device("cpu")
model = VAE(latent_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print("Model başarıyla oluşturuldu!")

Model başarıyla oluşturuldu!


In [17]:
def loss_function(recon_x, x, mu, logvar):
    # Görsel ne kadar benziyor? (MSE veya Binary Cross Entropy)
    BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    
    # Dağılım ne kadar düzenli? (KL Divergence)
    # Bu kısım modelin yeni ve anlamlı yüzler üretmesini sağlar
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return BCE + KLD

In [ ]:
num_epochs = 15
model.train()

for epoch in range(num_epochs):
    train_loss = 0
    for batch_idx, data in enumerate(dataloader):
        data = data.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        recon_batch, mu, logvar = model(data)
        loss = loss_function(recon_batch, data, mu, logvar)
        
        # Backward pass
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        
    print(f'Epoch {epoch+1}, Average Loss: {train_loss / len(dataloader.dataset):.4f}')

Epoch 1, Average Loss: 340.2547


In [ ]:
model.eval()
with torch.no_grad():
    # Rastgele gürültüden (noise) yüz üretme
    sample = torch.randn(64, 128).to(device) # 128 = latent_dim
    sample = model.decoder_input(sample).view(-1, 256, 4, 4)
    generated_images = model.decoder(sample).cpu()

# Üretilen yüzleri görselleştirme
plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(generated_images[i].permute(1, 2, 0))
    plt.axis('off')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Eğitim kaybını görselleştirme
plt.figure(figsize=(10, 5))
# num_epochs kadar bir liste oluşturup görselleştiriyoruz
plt.plot(range(1, num_epochs + 1), [train_loss / len(dataloader.dataset) for _ in range(num_epochs)], 
         marker='o', ls='-', color='g', label='Total Loss')
plt.title('VAE Eğitim Performansı (Optimized)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Modeli .pth dosyası olarak kaydet
torch.save(model.state_dict(), 'vae_celeba_final.pth')
print("Model 'vae_celeba_final.pth' adıyla kaydedildi. Sağ taraftaki Output kısmından indirebilirsin.")